# Tâche 2 — Estimation d'âge

Notebook généré à partir du script `Tache_2_Estimation_Age.py` (Auteur : Christ-Amour Kakpo).
Ce notebook contient le pipeline minimal PyTorch pour l'estimation d'âge : dataset, modèle, entraînement et génération de CSV de soumission.

## Installation
Installez les dépendances recommandées dans un virtualenv :
```bash
pip install torch torchvision timm pandas scikit-learn pillow tqdm
```

In [ ]:
# Imports principaux
import os
from pathlib import Path
import random
import numpy as np
from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torchvision.models as models
import pandas as pd

print('Imports OK')

In [ ]:
# ---------- Utilities (seed, list_images) ----------
def seed_everything(seed=42):
    import torch.backends.cudnn as cudnn
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    try:
        torch.cuda.manual_seed_all(seed)
    except Exception:
        pass
    # Pour Mac MPS (PyTorch 2.x) - tentative
    try:
        if torch.backends.mps.is_available():
            try:
                torch.mps.manual_seed(seed)
            except Exception:
                pass
    except Exception:
        pass
    cudnn.deterministic = True
    cudnn.benchmark = False
    print(f'✅ Seeds set to {seed}')


def list_images(folder, exts={'.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG'}):
    p = Path(folder)
    if not p.exists():
        return []
    return [str(x) for x in p.rglob('*') if x.suffix in exts]

print('Utilities defined')

In [ ]:
# ---------- Model: AgeEstimator ----------
class AgeEstimator(nn.Module):
    def __init__(self, n_bins=101, embedding_dim=512, pretrained=True):
        super().__init__()
        r = models.resnet18(weights=models.ResNet18_Weights.DEFAULT if pretrained else None)
        self.backbone = nn.Sequential(*list(r.children())[:-1])
        in_dim = r.fc.in_features
        self.fc = nn.Linear(in_dim, embedding_dim)
        self.cls = nn.Linear(embedding_dim, n_bins)
        self.reg = nn.Linear(embedding_dim, 1)

    def forward(self, x):
        x = self.backbone(x)
        x = x.view(x.size(0), -1)
        emb = self.fc(x)
        emb = nn.functional.relu(emb)
        logits = self.cls(emb)
        reg = self.reg(emb).squeeze(1)
        return logits, reg

print('Model defined')

In [ ]:
# ---------- Training & evaluation helpers ----------
def train_one_epoch(model, loader, opt, device, epoch, scheduler=None, lambda_reg=1.0):
    model.train()
    loss_ce = nn.CrossEntropyLoss()
    loss_mse = nn.MSELoss()
    total_loss = 0.0
    total_mae = 0.0
    total = 0

    for imgs, ages, _ in tqdm(loader, desc=f'"Train ep {epoch}"'):
        imgs = imgs.to(device)
        ages = ages.to(device)

        mask = (ages >= 0)
        if mask.sum() == 0:
            continue

        logits, reg = model(imgs)
        ages_int = ages.long().clamp(0, 100)
        loss1 = loss_ce(logits, ages_int)
        loss2 = loss_mse(reg[mask], ages[mask])
        loss = loss1 + lambda_reg * loss2

        opt.zero_grad()
        loss.backward()
        opt.step()

        if scheduler: scheduler.step()

        with torch.no_grad():
            preds = (torch.softmax(logits, dim=1) * torch.arange(0, logits.size(1), device=device).float()).sum(dim=1)
            mae = torch.abs(preds[mask] - ages[mask]).sum().item()

        total_loss += loss.item() * imgs.size(0)
        total_mae += mae
        total += mask.sum().item()

    avg_loss = total_loss / (len(loader.dataset) + 1e-9)
    avg_mae = total_mae / (total + 1e-9)
    return avg_loss, avg_mae

def evaluate(model, loader, device):
    model.eval()
    total_mae = 0.0
    total = 0
    preds_all = []
    files_all = []

    with torch.no_grad():
        for imgs, ages, paths in tqdm(loader, desc='Eval'):
            imgs = imgs.to(device)
            ages = ages.to(device, dtype=torch.float32)

            logits, reg = model(imgs)
            probs = torch.softmax(logits, dim=1)
            bins = torch.arange(0, logits.size(1), device=device, dtype=torch.float32)
            preds_cls = (probs * bins).sum(dim=1)
            preds = 0.5 * preds_cls + 0.5 * reg

            mask = (ages >= 0)
            if mask.sum() > 0:
                total_mae += torch.abs(preds[mask] - ages[mask]).sum().item()
                total += mask.sum().item()

            preds = preds.cpu().numpy()
            for p, pred in zip(paths, preds):
                preds_all.append(float(pred))
                files_all.append(Path(p).name)

    avg_mae = total_mae / (total + 1e-9) if total > 0 else None
    return files_all, preds_all, avg_mae

print('Training/eval helpers defined')

In [ ]:
# ---------- Main: interactive helper to run pipeline in notebook ----------
def run_pipeline(mode='eval', data_dir='data', out_dir='outputs', batch_size=32, epochs=5, lr=1e-4, lambda_reg=1.0, pretrained=True, resume='', seed=42, cpu=True):
    seed_everything(seed)
    device = torch.device('cpu' if cpu or not torch.cuda.is_available() else 'cuda')

    data_dir = Path(data_dir)
    train_dir = data_dir / 'train'
    test_dir = data_dir / 'test'
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    (out_dir / 'checkpoints').mkdir(exist_ok=True)
    (out_dir / 'submissions').mkdir(exist_ok=True)

    train_tf = T.Compose([
        T.Resize((160,160)),
        T.CenterCrop(128),
        T.RandomHorizontalFlip(),
        T.ColorJitter(0.1,0.1,0.1,0.05),
        T.ToTensor(),
        T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
    ])
    val_tf = T.Compose([
        T.Resize((128,128)),
        T.ToTensor(),
        T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
    ])

    train_files = list_images(train_dir)
    test_files = list_images(test_dir)
    print(f'Found {len(train_files)} train images, {len(test_files)} test images')
    if len(train_files) == 0 and mode=='train':
        print('No train images found; abort.')
        return

    train_ds = AgeDataset(train_files, transform=train_tf, is_train=True)
    val_subset_size = min(2000, len(train_ds))
    val_indices = np.random.choice(len(train_ds), val_subset_size, replace=False) if len(train_ds)>0 else []
    val_ds = torch.utils.data.Subset(train_ds, val_indices) if len(val_indices)>0 else train_ds

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=4)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, num_workers=4)
    test_ds = AgeDataset(test_files, transform=val_tf, is_train=False)
    test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False, num_workers=4)

    model = AgeEstimator(n_bins=101, pretrained=pretrained).to(device)

    if resume and Path(resume).exists():
        ck = torch.load(resume, map_location=device)
        model.load_state_dict(ck['model'])
        print('Loaded checkpoint', resume)

    if mode == 'train':
        params = model.parameters()
        opt = torch.optim.AdamW(params, lr=lr)
        scheduler = None
        for epoch in range(1, epochs+1):
            loss, train_mae = train_one_epoch(model, train_loader, opt, device, epoch, scheduler, lambda_reg=lambda_reg)
            print(f'Epoch {epoch} | loss: {loss:.4f} | train_mae(est): {train_mae:.3f}')
            ckpt = out_dir / 'checkpoints' / f'ckpt_ep{epoch}.pth'
            torch.save({'model': model.state_dict(), 'epoch': epoch}, ckpt)

    if mode == 'eval':
        if resume and Path(resume).exists():
            ck = torch.load(resume, map_location=device)
            model.load_state_dict(ck['model'])
            print('Loaded checkpoint', resume)
        print('Running inference on test set...')
        files, preds, _ = evaluate(model, test_loader, device)
        preds_int = [int(max(0, min(120, round(p)))) for p in preds]
        out_rows = [{'image': f, 'age': a} for f,a in zip(files, preds_int)]
        out_df = pd.DataFrame(out_rows)
        out_csv = out_dir / 'submissions' / 'task2_age_submission.csv'
        out_df.to_csv(out_csv, index=False)
        print('Saved submission to', out_csv)

print('run_pipeline helper defined')

In [ ]:
# Exemple d'utilisation interactive (décommentez et adaptez)
# run_pipeline(mode='eval', data_dir='data/dataset_tache_2/dataset_tache_2', out_dir='outputs/tache2', batch_size=32, cpu=True)

# Ou utilisez un Namespace pour appeler directement main(args) si vous préférez :
# from argparse import Namespace
# args = Namespace(mode='eval', data_dir='data/dataset_tache_2/dataset_tache_2', out_dir='outputs/tache2', batch_size=32, epochs=5, lr=1e-4, lambda_reg=1.0, pretrained=True, resume='', seed=42, cpu=True)
# main(args)

print('Notebook ready. Exécutez run_pipeline(...) selon vos besoins.')